In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 110})

def etiquetar_barras(ax):
    for cont in ax.containers:
        ax.bar_label(cont, fontsize=9, padding=2)

In [ ]:
raw = pd.read_csv('Analisis_GAC_Full7agosto_2026_Leads_por_Canal.csv', header=None)
canales = raw.iloc[4:10, [0] + list(range(1, 31))].copy()
canales.columns = ['Canal'] + list(pd.date_range('2024-01-01', periods=30, freq='MS').strftime('%Y-%m'))
canales['Canal'] = canales['Canal'].str.strip()

# --- Tratamiento de valores nulos ---
canales = canales.replace({'-': np.nan})              # guiones = sin dato
for c in canales.columns[1:]:
    canales[c] = pd.to_numeric(canales[c], errors='coerce')
print("Nulos antes de limpiar:", canales.isnull().sum().sum())
canales[canales.columns[1:]] = canales[canales.columns[1:]].fillna(0)  # sin leads ese mes = 0
print("Nulos después de limpiar:", canales.isnull().sum().sum())

leads_canal = canales.melt(id_vars='Canal', var_name='Periodo', value_name='Leads')
leads_canal.to_csv('leads_por_canal_clean.csv', index=False)

In [ ]:
raw_sdc = pd.read_csv('Analisis_GAC_Full7agosto_2026_SDC.csv', header=None)

# Filas 5-9 = Generadas, Aprobadas, Condicionadas, Rechazadas, Formalizadas
# Columnas 1-31 = Enero 2024 a Julio 2026 (31 meses)
filas = {'Generadas': 5, 'Aprobadas': 6, 'Condicionadas': 7, 'Rechazadas': 8, 'Formalizadas': 9}
periodo = pd.date_range('2024-01-01', periods=31, freq='MS').strftime('%b-%y')

sdc = pd.DataFrame({'Periodo': periodo})
for nombre, fila in filas.items():
    # reset_index(drop=True) evita que pandas desalinee los datos por índice
    valores = raw_sdc.iloc[fila, 1:32].astype(str).str.strip().reset_index(drop=True)
    sdc[nombre] = pd.to_numeric(valores, errors='coerce')

pct_aprob = raw_sdc.iloc[11, 1:32].astype(str).str.replace('%', '').str.strip().reset_index(drop=True)
pct_form = raw_sdc.iloc[12, 1:32].astype(str).str.replace('%', '').str.strip().reset_index(drop=True)
sdc['Pct_Aprobacion'] = pd.to_numeric(pct_aprob, errors='coerce')
sdc['Pct_Formalizacion'] = pd.to_numeric(pct_form, errors='coerce')

# --- Tratamiento de valores nulos ---
print("Nulos antes de limpiar:\n", sdc.isnull().sum())
sdc = sdc.dropna().reset_index(drop=True)
print("Nulos después de limpiar:\n", sdc.isnull().sum())

sdc.to_csv('sdc_clean.csv', index=False)
sdc.head()

Variables categóricas

In [ ]:
Tabla_freq = leads_canal.groupby('Canal')['Leads'].sum().reset_index()
Tabla_freq.columns = ['Canal', 'count']
Filtro_index = Tabla_freq.sort_values('count', ascending=False).set_index('Canal')

n_total = int(Filtro_index['count'].sum())
Filtro_index.columns = [f'Leads totales acumulados (n = {n_total})']

ax = Filtro_index.plot(kind='bar', width=0.7, color='darkgreen', figsize=(8,4), legend=True, rot=30)
plt.title(f'Leads Totales por Canal de Origen (2024-2026) — Total = {n_total:,}')
plt.xlabel('Canal'); plt.ylabel('Leads totales acumulados')
etiquetar_barras(ax)
plt.show()

In [ ]:
Tabla_freq = Tabla_freq.sort_values('Canal')
total_leads = int(Tabla_freq['count'].sum())

def autopct_con_valor(valores):
    def inner(pct):
        val = int(round(pct/100.*sum(valores)))
        return f'{pct:.1f}%\n({val:,})'
    return inner

fig, ax = plt.subplots(figsize=(7.5,7))
ax.pie(Tabla_freq['count'], labels=Tabla_freq['Canal'],
       autopct=autopct_con_valor(Tabla_freq['count']), shadow=True, pctdistance=0.75)
ax.set_title(f'Distribución % de Leads por Canal (Total = {total_leads:,} leads)')
plt.show()

In [ ]:
col = 'Pct_Aprobacion'
n = sdc[col].count()
Max = sdc[col].max(); Min = sdc[col].min()
R = Max - Min
ni = int(round(1 + 3.32*np.log10(n)))
print(f"n={n}, Max={Max}, Min={Min}, R={R}, ni={ni}")

fig, ax = plt.subplots(figsize=(7,4))
counts, bins, patches = ax.hist(sdc[col], bins=ni, color='purple', edgecolor='black', label=f'n = {n} meses')
ax.bar_label(patches, labels=[int(c) for c in counts], padding=2, fontsize=9)
ax.set_title(f'Distribución del % de Aprobación de Crédito en {ni} rangos (Sturges)')
ax.set_xlabel('% de Aprobación — rangos de valor, NO representa meses en orden cronológico')
ax.set_ylabel('Frecuencia (cantidad de meses)')
ax.legend()
plt.show()

# Tabla de referencia
sdc['Pct_Aprobacion_clase'] = pd.cut(sdc['Pct_Aprobacion'], bins=ni)
print(sdc[['Periodo','Pct_Aprobacion','Pct_Aprobacion_clase']].sort_values('Pct_Aprobacion').to_string(index=False))

In [ ]:
col = 'Formalizadas'
n = sdc[col].count()
Max = sdc[col].max(); Min = sdc[col].min()
R = Max - Min
ni = int(round(1 + 3.32*np.log10(n)))
i = R/ni
print(f"n={n}, Max={Max}, Min={Min}, R={R}, ni={ni}, i={round(i,2)}")

intervalos = np.linspace(Min-1, Max+1, ni+1)
categorias = [f"C{k+1}\n({int(intervalos[k])}-{int(intervalos[k+1])})" for k in range(ni)]
sdc[f'{col}_clase'] = pd.cut(sdc[col], bins=intervalos, labels=categorias)

Tabla_freq = sdc[f'{col}_clase'].value_counts().sort_index().reset_index()
Tabla_freq.columns = ['Clase', 'count']
Filtro_index = Tabla_freq.set_index('Clase')
Filtro_index.columns = [f'Frecuencia (n = {n} meses)']

ax = Filtro_index.plot(kind='bar', width=0.8, color='crimson', figsize=(9,4), legend=True, rot=0)
plt.title(f'Créditos Formalizados agrupados en {ni} clases (Sturges, amplitud≈{round(i)})')
plt.xlabel('Rango de Créditos Formalizados'); plt.ylabel('Frecuencia (meses)')
etiquetar_barras(ax)
plt.show()

# Tabla de referencia
print(sdc[['Periodo', col, f'{col}_clase']].sort_values(col).to_string(index=False))